# Step 2 — Sample-Level Diagnostic Plots

## The problem this step solves

You now have count matrices, but you do not yet know what a "good" cell looks like in *this particular dataset*. Every tissue, protocol, and sequencing run has a different distribution of cell quality metrics. Setting the wrong filtering thresholds in Step 3 has serious consequences:

- **Too permissive:** dead cells and empty droplets stay in. Dead cells cluster together and look like a cell type. Empty droplets add noise that blurs cluster boundaries.
- **Too aggressive:** you remove a whole population of rare, small cells (like adipose stem cells) because they have naturally low gene counts. The paper's key finding about MSCs depends on capturing these rare populations — SVF enrichment was specifically used because ASCs are too rare to find without it.

This step visualizes the per-cell quality metric distributions for each sample *before* applying any filter. The plots guide threshold decisions that are specific to each sample rather than one-size-fits-all.

## Why this matters for the Yang et al. paper

The study used **stromal vascular fraction (SVF) enrichment** for adipose tissues rather than bulk dissociation. SVF is everything except mature fat cells — it is enriched in adipose stem cells (ASCs), immune cells, and endothelial cells. Mature adipocytes, which are already well-characterized, are depleted. This enrichment means the cell quality distributions in this dataset are different from a naive adipose tissue dissociation: the "normal" cell may have lower UMI counts than you would expect if you applied thresholds from a different protocol.

Looking at the distribution before filtering lets you set thresholds that match the actual biology of these enriched samples rather than generic defaults. The paper used `nFeature_RNA > 200`, `nCount_RNA > 500`, and `percent_mt < 30` — the 30% mitochondrial threshold is deliberately permissive because adipose tissue, especially from exercised animals, has elevated mitochondrial content from metabolically active beige adipocytes.

## What goes wrong if you skip this

If you apply generic thresholds without inspecting distributions, you apply the same cutoff to every sample. A skeletal muscle sample (which has high mitochondrial content from oxidative muscle fibers) would be over-filtered with an adipose-tissue threshold. A sample with naturally low gene counts (small stromal cells) would be eliminated. The distributions are the data — look at them first.

## What this notebook does

For each sample:
1. Load the 10x count matrix from the CellRanger `filtered_feature_bc_matrix/` directory
2. Compute three per-cell QC metrics: gene count, UMI count, mitochondrial fraction
3. Generate a **violin plot** — shows the full distribution; where is the low-quality tail?
4. Generate **scatter plots** — reveals correlations between metrics that catch doublets and dying cells
5. Run PCA and produce an **elbow plot** — tells you how many PCs carry real signal vs. noise

All plots saved to `seurat_diagnostic_plots/`.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

ModuleNotFoundError: No module named 'scanpy'

## Configuration

- `species` — controls which gene name prefix is used to identify mitochondrial genes (`mt-` for mouse, `MT-` for human)
- `norm_mode` — `regular` scales to the per-cell median count then log-transforms; `sctransform` uses a regularized negative binomial regression that is more robust to technical variation in sequencing depth but is slower
- `pc_num` — number of PCs to compute and display on the elbow plot; this is also used in Step 3 for DoubletFinder and neighbor graph construction

In [4]:
cellranger_input_file = "cellranger_manifest.txt"
species = "mouse"       # "mouse" or "human"
norm_mode = "regular"   # "regular" or "sctransform"
pc_num = 20
target_folder = Path(".")

plot_folder = target_folder / "seurat_diagnostic_plots"
plot_folder.mkdir(exist_ok=True)

## Load the manifest

In [ ]:
manifest = pd.read_csv(cellranger_input_file, sep="\t")
print(f"Loaded manifest with {len(manifest)} samples")
manifest.head()

## Per-sample diagnostic loop

### The three QC metrics

Every cell receives three quality metrics that together distinguish real cells from artifacts:

| Metric | What it measures | Biological interpretation |
|--------|-----------------|---------------------------|
| `nFeature_RNA` | Number of distinct genes detected | Proxy for cell complexity. Very low values indicate empty droplets or dead cells; very high values can indicate doublets. |
| `nCount_RNA` | Total UMI count (transcripts captured) | Proxy for sequencing depth per cell. Should correlate tightly with `nFeature_RNA`; a cell with high counts but few genes is suspicious. |
| `percent_mt` | Fraction of UMIs mapping to mitochondrial genes | Key viability indicator. Healthy cells use mitochondria but devote most transcription to cytoplasmic mRNA. When a cell is stressed or lysing, cytoplasmic mRNA leaks out first, leaving mitochondrial transcripts disproportionately represented. A high `percent_mt` (typically > 20–25% depending on tissue) signals a dying cell. |

The mitochondrial gene prefix differs by species: `mt-` in mouse (lowercase), `MT-` in human (uppercase). Using the wrong prefix will silently set `percent_mt` to zero for every cell, making it impossible to filter dying cells downstream.

### The three plots

- **Violin plot** — shows the full distribution of each metric across all cells in the sample. The shape of the violin tells you whether thresholds should be symmetric (roughly normal distribution) or asymmetric (long tails). Comparing violins across samples quickly reveals whether one sample has a shifted distribution that warrants a different threshold.
- **Scatter plots** — the `nCount_RNA` vs `nFeature_RNA` scatter should show a tight positive correlation; outliers (high counts, few genes) are doublet candidates. The `nCount_RNA` vs `percent_mt` scatter should show that low-count barcodes have high mitochondrial fractions — the classic signature of empty droplets and dying cells.
- **Elbow plot** — after PCA, the variance explained by each PC drops off. The "elbow" is the point where additional PCs contribute little new variance and mostly capture noise. The number of PCs at the elbow (typically 10–30) is used as `pc_num` in Step 3 and the integration step.

In [ ]:
mt_prefix = "mt-" if species == "mouse" else "MT-"

for _, row in manifest.iterrows():
    lib_id = row["library_ID"]
    data_path = Path(row["folder_path"]) / "filtered_feature_bc_matrix"
    print(f"\nProcessing: {lib_id}")

    # ----------------------------------------------------------------
    # Load count matrix
    # scanpy reads the 10x MEX format (matrix.mtx.gz, barcodes.tsv.gz,
    # features.tsv.gz) produced by CellRanger. We prefix barcodes with
    # the library ID to keep them unique when samples are later merged.
    # ----------------------------------------------------------------
    adata = sc.read_10x_mtx(data_path, var_names="gene_symbols", cache=False)
    adata.obs_names = [f"{lib_id}_{bc}" for bc in adata.obs_names]
    adata.var_names_make_unique()
    # var_names_make_unique() deduplicates by appending a suffix to duplicates:

    # ----------------------------------------------------------------
    # Compute QC metrics
    # sc.pp.calculate_qc_metrics fills in:
    #   obs['n_genes_by_counts']  → nFeature_RNA equivalent 
    #       how many distinct genes did we detect in the cell.  How many genes had at least umi 
    #   obs['total_counts']       → nCount_RNA equivalent
    #   obs['pct_counts_mt']      → percent_mt equivalent

    # n_genes_by_counts counts how many genes had at least 1 UMI — it's a count of nonzero entries.
    # total_counts sums how many molecules total — it adds up all the UMI values.

    # A cell could have high total_counts but low n_genes_by_counts if it's expressing a smal
    #  number of genes very highly — like a cell pumping out enormous amounts of one or two 
    # transcripts. That's suspicious and is one signature of a doublet or an artifact. The 
    # scatter plot between the two is specifically looking for cells that fall off the diagonal 
    # for exactly this reason.
    # ----------------------------------------------------------------
    adata.var["mt"] = adata.var_names.str.startswith(mt_prefix)
    sc.pp.calculate_qc_metrics(
        adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
    )

    # ================================================================
    # Plot 1 — Violin plots of QC metrics
    # Each panel shows the distribution across all barcodes in the
    # sample. Overlapping jitter points are suppressed (strip=False)
    # to keep the plot readable for large samples.
    # ================================================================
    fig, axes = plt.subplots(1, 3, figsize=(9, 4))
    fig.suptitle(lib_id, fontsize=11)

    for ax, (col, label) in zip(axes, [
        ("n_genes_by_counts", "nFeature_RNA"),
        ("total_counts",      "nCount_RNA"),
        ("pct_counts_mt",     "percent.mt"),
    ]):
        ax.violinplot(adata.obs[col], positions=[0], showmedians=True)
        ax.set_xticks([])
        ax.set_ylabel(label)

    plt.tight_layout()
    fig.savefig(plot_folder / f"{lib_id}_violin.png", dpi=150)
    plt.show()
    plt.close(fig)

    # ================================================================
    # Plot 2 — Scatter plots of QC metric correlations
    # Left:  nCount vs percent_mt
    #   Cells in the bottom-right are healthy (high counts, low %mt).
    #   Cells in the top-left are dying (low counts, high %mt).
    # Right: nCount vs nFeature
    #   Should be a tight diagonal. Outliers above the line (high
    #   counts relative to genes) are likely doublets.
    # ================================================================
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    fig.suptitle(lib_id, fontsize=11)

    axes[0].scatter(
        adata.obs["total_counts"], adata.obs["pct_counts_mt"],
        s=1, alpha=0.3, rasterized=True
    )
    axes[0].set_xlabel("nCount_RNA")
    axes[0].set_ylabel("percent.mt")

    axes[1].scatter(
        adata.obs["total_counts"], adata.obs["n_genes_by_counts"],
        s=1, alpha=0.3, rasterized=True
    )
    axes[1].set_xlabel("nCount_RNA")
    axes[1].set_ylabel("nFeature_RNA")

    plt.tight_layout()
    fig.savefig(plot_folder / f"{lib_id}_scatter.png", dpi=150)
    plt.show()
    plt.close(fig)

    # ================================================================
    # Normalize, find variable features, run PCA
    # ================================================================
    if norm_mode == "regular":
        # Normalize each cell to the median total count across all
        # cells, then log1p-transform. This puts cells on a comparable
        # scale without distorting relative expression levels.
        target_sum = float(np.median(adata.obs["total_counts"]))
        sc.pp.normalize_total(adata, target_sum=target_sum)
        # divide by total number of umis, and multiply by target sum to avoid decimals
        sc.pp.log1p(adata)
        # because a gene has a high right skew - where a highly expressed gene has 5k counts, and low has count of 2,
        # a handful of highly expressed genes dominate everything — PCA, distance calculations, clustering.
    elif norm_mode == "sctransform":
        # SCTransform (Hafemeister & Satija 2019) fits a regularized
        # negative binomial model per gene to regress out sequencing
        # depth, producing Pearson residuals. It is more robust than
        # log-normalization when depth varies widely across cells.
        # The closest scanpy equivalent is: normalize → log1p → scale,
        # or use the external `sctransform` package via rpy2.
        # Here we use the scanpy approximation for portability.
        sc.pp.normalize_total(adata)
        sc.pp.log1p(adata)
        sc.pp.scale(adata, max_value=10)
        # PCA finds directions of maximum variance. Without scaling, Actb dominates PC1 simply because it has high absolute values — not because it's biologically informative. A rare marker that varies between cell types but has low absolute counts barely contributes to any PC.
        # # scale standardizes each gene (column) to mean=0, standard deviation=1 — the standard z-score transform:

# 

    # Identify highly variable genes — the ~2,000 genes with the most
    # biological variance relative to their mean expression. PCA is
    # computed only on these genes to reduce noise from housekeeping
    # genes that are uniformly expressed across all cell types.
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    sc.tl.pca(adata, n_comps=pc_num, use_highly_variable=True)

    # ================================================================
    # Plot 3 — PCA elbow plot
    # Variance explained by each PC. The elbow (where the curve
    # flattens) indicates how many PCs capture real biological signal
    # vs. noise. The value chosen here propagates to Step 3
    # (DoubletFinder, neighbor graph) and Step 5 (integration).
    # ================================================================
    variance_ratio = adata.uns["pca"]["variance_ratio"]

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(range(1, len(variance_ratio) + 1), variance_ratio, "o-", ms=4)
    ax.set_xlabel("PC")
    ax.set_ylabel("Fraction of variance explained")
    ax.set_title(f"{lib_id} — PCA elbow")
    plt.tight_layout()
    fig.savefig(plot_folder / f"{lib_id}_elbow.png", dpi=150)
    plt.show()
    plt.close(fig)

print("\nDone. Plots saved to:", plot_folder)

## Interpreting the plots and deciding thresholds

After reviewing the plots, record the per-sample thresholds you plan to use in Step 3. There is no universal cutoff — the right values depend on the tissue, protocol, and sequencing depth of each sample.

### General guidelines

**nFeature_RNA (genes per cell)**
- Lower bound: removes empty droplets. A common starting point is 200–500 genes. Look at where the lower tail of the violin separates from the bulk of the distribution.
- Upper bound: removes doublets (though DoubletFinder in Step 3 handles this more precisely). A rough heuristic is 2–3× the median.

**nCount_RNA (UMIs per cell)**
- Correlated with nFeature; often filtered together. An upper bound of ~25,000–50,000 removes extreme outliers likely to be doublets.

**percent_mt**
- The appropriate threshold is tissue-dependent. Metabolically active tissues (e.g., skeletal muscle, cardiomyocytes) have naturally higher mitochondrial content. For adipose tissue and most dissociated cell types, 10–25% is a common range. Muscle may tolerate higher values.
- The scatter plot is especially useful here: if the low-count / high-mt cluster is clearly separated from the main population, the threshold is visually obvious.

**PCA elbow**
- Identify the PC where the curve transitions from steep to flat. Use that number as `pc_num` in Step 3. If different samples suggest different elbows, choose a conservative value that works across all of them (usually 15–30).

These thresholds feed directly into the `nFeature`, `nCount`, and `percent.mt` cutoffs in `sample_level_processing.R` (Step 3).